In [ ]:
#| hide
from drona.core import *
from drona.rounds import *
from aidialog.ipynb import read_ipynb, write_ipynb

# drona

> Train agents to choose and use the right tools.

An agent copies the tool calls that it finds in its context. Drona controls that context.

Drona reads the session logs of Ramabana, Claude Code, and Codex. Drona gives a score to the tool
calls in each log. A person then reviews one conversation. Drona makes the first history of a new
session from that conversation.

```sh
pip install drona
```

## 1. A session log

Ramabana writes one JSON line for each turn that ends. The file is
`~/.config/ramabana/agent-history.jsonl`. The example session below has two turns. In the first
turn, the agent uses the wrong tool. In the second turn, the agent uses the correct tool.

In [ ]:
import json, tempfile
from pathlib import Path

ASK = 'Use fossick to research the AnswerDotAI llmdojo github repository'
wrong_turn = {'session': 'sess-9f2a', 'state': 'complete', 'prompt': ASK,
          'reply': 'That search did not give the source files.',
          'activity': [
              {'action_id': 'a0', 'tool': 'web_search', 'ok': True,
               'args': {'query': 'llmdojo answerdotai'}, 'detail': '10 results, mostly forks.'},
              {'action_id': 'a1', 'tool': 'read_url', 'ok': False,
               'args': {'url': 'https://llmdojo.dev'}, 'detail': '404'}]}
right_turn = {'session': 'sess-9f2a', 'state': 'complete', 'prompt': ASK,
        'reply': 'FOSSICK read the repository. Its files answer the question.',
        'activity': [
            {'action_id': 'b0', 'tool': 'run_shell', 'ok': True,
             'args': {'command': 'fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo'},
             'detail': '# llmdojo\nAn agent copies the tool calls that it finds in its context.'}]}

tmp = Path(tempfile.mkdtemp())
archive = tmp/'agent-history.jsonl'
archive.write_text('\n'.join(json.dumps(t) for t in (wrong_turn, right_turn)) + '\n')
print(archive.read_text()[:120], '…')

{"session": "sess-9f2a", "state": "complete", "prompt": "Use fossick to research the AnswerDotAI llmdojo github reposito …


Drona reads this file after the session ends. Drona does not monitor a live process.

Ramabana gives each turn a `state` of `complete`, `failed`, or `abandoned`. Drona keeps only the
turns with the state `complete`. A turn that a person stopped is not a good example.

## 2. Score the tool calls

`assess_turn` examines the tool calls of one turn. It does not examine the text of the reply. Each
finding gives the index of one tool call.

In [ ]:
assess_turn(wrong_turn)

Assessment(score=80, calls=2, findings=(Finding(kind='route', tool='web_search', index=0, message='Use fossick read-gh-repo as the first repository research call.'),))

The prompt gives the name of a repository. The prompt also gives the name of the tool that reads a
repository. Therefore the first tool call must be that tool.

The call to `web_search` decreases the score by 20 points. The failed call to `read_url` does not
decrease the score again. The sequence of the tool calls is the fault, not the failure. The second
turn uses the correct tool.

In [ ]:
assess_turn(right_turn)

Assessment(score=100, calls=1, findings=())

```sh
drona --session latest
```

## 3. Capture the session

`capture` makes an Aidialog notebook from the log. The notebook starts with a review note, which
Drona marks as skipped. After the note, the notebook has one prompt message for each turn. Each
prompt message contains the tool calls and the results of that turn. Drona puts the score in the
metadata.

In [ ]:
review = capture(tmp/'llmdojo.ipynb', history=archive)
dlg = read_ipynb(review)
print(dlg.meta['drona']['score'], dlg.meta['drona']['status'])
for m in dlg: print(f'{m.msg_type:7} skipped={m.skipped}  {str(m.content)[:52]!r}')

80 review
note    skipped=1  '# Drona review\n\nEdit this dialog in Leela. Delete th'
prompt  skipped=0  'Use fossick to research the AnswerDotAI llmdojo gith'
prompt  skipped=0  'Use fossick to research the AnswerDotAI llmdojo gith'


## 4. Review the notebook

A person must do this step.

1. Open the notebook in Leela.
2. Delete the incorrect tool calls.
3. Delete all private data.
4. Keep the tool calls that a later model must copy.

```sh
leela training
```

When you delete a message, Leela marks that message as skipped. In this example, delete the first
turn. `assess_turn` gave a finding for that turn.

In [ ]:
dlg[1].skipped = 1
write_ipynb(dlg, review)
for m in read_ipynb(review): print(f"{m.msg_type:7} {('kept', 'dropped')[m.skipped]:8} {str(m.content)[:44]!r}")

note    dropped  '# Drona review\n\nEdit this dialog in Leela. D'
prompt  dropped  'Use fossick to research the AnswerDotAI llmd'
prompt  kept     'Use fossick to research the AnswerDotAI llmd'


## 5. Accept the round

`accept` records the name of the reviewer, marks the notebook as accepted, and writes the history to
a JSON file next to the notebook. Drona compiles a round for a host only after a reviewer accepts
that round.

In [ ]:
compiled = accept(review, 'Karthik')
meta = json.loads(compiled.read_text())['meta']
print({k: meta[k] for k in ('status', 'reviewer', 'accepted_version')})

{'status': 'accepted', 'reviewer': 'Karthik', 'accepted_version': '0.0.1:2'}


## 6. Start the next session

An accepted round compiles to three formats.

The first format is Urai history. Give it to a chat. The history has no call to `web_search`, because the
reviewer deleted that turn. Only the correct tool call is left.

In [ ]:
for m in warm_start(review):
    shows = str(m.get('content')) or ' '.join(c.name for c in m.get('tool_calls', []))
    print(f"{m['role']:9} | {shows[:52]}")

user      | Use fossick to research the AnswerDotAI llmdojo gith
assistant | run_shell
tool      | # llmdojo
An agent copies the tool calls that it fin
assistant | FOSSICK read the repository. Its files answer the qu


The second format is a bootstrap prompt. Use this format for a host that does not accept a prepared
history on its command line. Ramabana is such a host. Drona shortens each tool result to 600
characters, because Ramabana uses the same limit when it replays a turn.

In [ ]:
print(bootstrap_prompt(review))

A reviewer approved the Drona round below. Use the same tools, in the same sequence.

User: Use fossick to research the AnswerDotAI llmdojo github repository

Assistant tool: run_shell({"command": "fossick read-gh-repo https://github.com/AnswerDotAI/llmdojo"})

Tool result: # llmdojo
An agent copies the tool calls that it finds in its context.

Assistant: FOSSICK read the repository. Its files answer the question.

Reply with exactly: DRONA_READY


`drona-start` prints two commands. The `--launch` option runs the two commands. The first command
sends the round as one bootstrap turn. The second command resumes the session that the first command
made.

```sh
drona-start training/llmdojo.ipynb --root /path/to/project --launch
```

In [ ]:
for name, cmd in start_round(review, root='/path/to/project').items(): print(name, cmd[:3])

bootstrap ['ramabana', '--root', '/path/to/project']
resume ['ramabana', '--root', '/path/to/project']


## Move a round between hosts

The third format is a host session file. All three hosts share the notebook format. Therefore you
can capture a session from one host, review that session one time, and start a new session on a
different host.

```sh
drona-capture training/round.ipynb                                     # ramabana
drona-capture training/round.ipynb --host claude --cwd /path/to/project
drona-capture training/round.ipynb --host codex  --cwd /path/to/project

drona-export training/round.ipynb ramabana --output training/round.txt
drona-export training/round.ipynb claude --cwd /path/to/project
drona-export training/round.ipynb codex --output training/items.json
```

The Claude export prints a session id. Claude Code can resume that session. The Codex export does
not make a session that you can resume, because llmsurgery does not have a public rollout writer.
You can still use the Codex items to examine a round and to build datasets.

## Develop

```sh
uv sync --all-extras --group dev
uv run nbdev-export
uv run nbdev-test
uv run nbdev-readme
uv run nbdev-clean
```